This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Object detection

### Single-stage vs. two-stage object detectors

#### Two-stage R-CNN detectors

#### Single-stage detectors

### Training a YOLO model from scratch

#### Downloading the COCO dataset

In [ ]:
# PyTorch: there's no keras.utils.get_file. We download and extract the COCO
# train2017 images and annotations with urllib + zipfile into a local cache dir.
import os
import urllib.request
import zipfile

cache_dir = os.path.expanduser("~/.keras/datasets")
os.makedirs(cache_dir, exist_ok=True)


def download_and_extract(url, name):
    zip_path = os.path.join(cache_dir, name + ".zip")
    out_dir = os.path.join(cache_dir, name)
    if not os.path.exists(out_dir):
        if not os.path.exists(zip_path):
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(out_dir)
    return out_dir


images_path = download_and_extract(
    "http://images.cocodataset.org/zips/train2017.zip", "coco"
)
annotations_path = download_and_extract(
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "annotations",
)

In [ ]:
import json

with open(f"{annotations_path}/annotations/instances_train2017.json", "r") as f:
    annotations = json.load(f)

# PyTorch: keras_hub.utils.coco_id_to_name isn't available, so we build the same
# COCO category-id -> name lookup directly from the annotation file's categories.
_coco_names = {c["id"]: c["name"] for c in annotations["categories"]}

def coco_id_to_name(label):
    return _coco_names.get(int(label), "background")

images = {image["id"]: image for image in annotations["images"]}

def scale_box(box, width, height):
    scale = 1.0 / max(width, height)
    x, y, w, h = [v * scale for v in box]
    x += (height - width) * scale / 2 if height > width else 0
    y += (width - height) * scale / 2 if width > height else 0
    return [x, y, w, h]

metadata = {}
for annotation in annotations["annotations"]:
    id = annotation["image_id"]
    if id not in metadata:
        metadata[id] = {"boxes": [], "labels": []}
    image = images[id]
    box = scale_box(annotation["bbox"], image["width"], image["height"])
    metadata[id]["boxes"].append(box)
    metadata[id]["labels"].append(annotation["category_id"])
    metadata[id]["path"] = images_path + "/train2017/" + image["file_name"]
metadata = list(metadata.values())

In [0]:
len(metadata)

In [0]:
min([len(x["boxes"]) for x in metadata])

In [0]:
max([len(x["boxes"]) for x in metadata])

In [0]:
max(max(x["labels"]) for x in metadata) + 1

In [0]:
metadata[435]

In [ ]:
[coco_id_to_name(x) for x in metadata[435]["labels"]]

In [0]:
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Rectangle

color_map = {0: "gray"}

def label_to_color(label):
    if label not in color_map:
        h, s, v = (len(color_map) * 0.618) % 1, 0.5, 0.9
        color_map[label] = hsv_to_rgb((h, s, v))
    return color_map[label]

def draw_box(ax, box, text, color):
    x, y, w, h = box
    ax.add_patch(Rectangle((x, y), w, h, lw=2, ec=color, fc="none"))
    textbox = dict(fc=color, pad=1, ec="none")
    ax.text(x, y, text, c="white", size=10, va="bottom", bbox=textbox)

def draw_image(ax, image):
    ax.set(xlim=(0, 1), ylim=(1, 0), xticks=[], yticks=[], aspect="equal")
    image = plt.imread(image)
    height, width = image.shape[:2]
    hpad = (1 - height / width) / 2 if width > height else 0
    wpad = (1 - width / height) / 2 if height > width else 0
    extent = [wpad, 1 - wpad, 1 - hpad, hpad]
    ax.imshow(image, extent=extent)

In [ ]:
sample = metadata[435]
ig, ax = plt.subplots(dpi=300)
draw_image(ax, sample["path"])
for box, label in zip(sample["boxes"], sample["labels"]):
    label_name = coco_id_to_name(label)
    draw_box(ax, box, label_name, label_to_color(label))
plt.show()

In [0]:
import random

metadata = list(filter(lambda x: len(x["boxes"]) <= 4, metadata))
random.shuffle(metadata)

#### Creating a YOLO model

In [ ]:
image_size = 448

# PyTorch: keras_hub's ResNet-50 backbone -> torchvision's ImageNet-pretrained
# resnet50 with the classifier head removed (we keep the conv feature extractor).
# keras_hub.layers.ImageConverter -> a torchvision transforms pipeline that
# resizes to image_size and applies the standard ImageNet normalization.
import torch
from torch import nn
import torchvision
from torchvision import transforms

resnet = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
)
# Feature extractor: everything up to (but not including) avgpool/fc.
# Output is a (N, 2048, H/32, W/32) feature map.
backbone = nn.Sequential(*list(resnet.children())[:-2])

preprocessor = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

In [ ]:
grid_size = 6
num_labels = 91

# PyTorch: the functional keras.Model becomes an nn.Module. Conv2d/Linear need
# explicit input sizes. The ResNet-50 feature map is (N, 2048, 14, 14) for a
# 448x448 input; the stride-2 3x3 conv brings it to (N, 512, 6, 6), which we
# flatten. The final softmax over class logits is applied here to mirror the
# book's output dict (the box-confidence channels stay raw).
class YOLO(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.conv = nn.Conv2d(2048, 512, kernel_size=3, stride=2)
        self.flatten = nn.Flatten()
        self.dense1 = nn.Linear(512 * 6 * 6, 2048)
        nn.init.xavier_normal_(self.dense1.weight)  # glorot_normal
        self.dropout = nn.Dropout(0.5)
        self.dense2 = nn.Linear(2048, grid_size * grid_size * (num_labels + 5))

    def forward(self, x):
        x = self.backbone(x)
        x = self.conv(x)
        x = self.flatten(x)
        x = torch.relu(self.dense1(x))
        x = self.dropout(x)
        x = self.dense2(x)
        # PyTorch: feature maps are NCHW, but we reshape to the book's
        # (grid, grid, num_labels + 5) layout to keep the loss/decoding code intact.
        x = x.reshape(-1, grid_size, grid_size, num_labels + 5)
        box_predictions = x[..., :5]
        class_predictions = torch.softmax(x[..., 5:], dim=-1)
        return {"box": box_predictions, "class": class_predictions}


model = YOLO(backbone)

In [ ]:
# PyTorch: there is no model.summary(); printing the module shows its layers.
print(model)

#### Readying the COCO data for the YOLO model

In [0]:
def to_grid(box):
    x, y, w, h = box
    cx, cy = (x + w / 2) * grid_size, (y + h / 2) * grid_size
    ix, iy = int(cx), int(cy)
    return (ix, iy), (cx - ix, cy - iy, w, h)

def from_grid(loc, box):
    (xi, yi), (x, y, w, h) = loc, box
    x = (xi + x) / grid_size - w / 2
    y = (yi + y) / grid_size - h / 2
    return (x, y, w, h)

In [0]:
import numpy as np
import math

class_array = np.zeros((len(metadata), grid_size, grid_size))
box_array = np.zeros((len(metadata), grid_size, grid_size, 5))

for index, sample in enumerate(metadata):
    boxes, labels = sample["boxes"], sample["labels"]
    for box, label in zip(boxes, labels):
        (x, y, w, h) = box
        left, right = math.floor(x * grid_size), math.ceil((x + w) * grid_size)
        bottom, top = math.floor(y * grid_size), math.ceil((y + h) * grid_size)
        class_array[index, bottom:top, left:right] = label

for index, sample in enumerate(metadata):
    boxes, labels = sample["boxes"], sample["labels"]
    for box, label in zip(boxes, labels):
        (xi, yi), (grid_box) = to_grid(box)
        box_array[index, yi, xi] = [*grid_box, 1.0]
        class_array[index, yi, xi] = label

In [ ]:
def draw_prediction(image, boxes, classes, cutoff=None):
    fig, ax = plt.subplots(dpi=300)
    draw_image(ax, image)
    for yi, row in enumerate(classes):
        for xi, label in enumerate(row):
            color = label_to_color(label) if label else "none"
            x, y, w, h = (v / grid_size for v in (xi, yi, 1.0, 1.0))
            r = Rectangle((x, y), w, h, lw=2, ec="black", fc=color, alpha=0.5)
            ax.add_patch(r)
    for yi, row in enumerate(boxes):
        for xi, box in enumerate(row):
            box, confidence = box[:4], box[4]
            if not cutoff or confidence >= cutoff:
                box = from_grid((xi, yi), box)
                label = classes[yi, xi]
                color = label_to_color(label)
                name = coco_id_to_name(label)
                draw_box(ax, box, f"{name} {max(confidence, 0):.2f}", color)
    plt.show()

draw_prediction(metadata[0]["path"], box_array[0], class_array[0], cutoff=1.0)

In [ ]:
# PyTorch: tf.data is replaced by a Dataset + DataLoader. Each item loads an
# image with PIL, runs it through the preprocessor (-> a CHW tensor), and pairs
# it with the box/class target arrays. The "box"/"class" dict label becomes a
# tuple unpacked the same way in the training loop.
from PIL import Image
from torch.utils.data import Dataset, DataLoader


class CocoDataset(Dataset):
    def __init__(self, metadata, box_array, class_array):
        self.metadata = metadata
        self.box_array = box_array
        self.class_array = class_array

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        image = Image.open(self.metadata[idx]["path"]).convert("RGB")
        x = preprocessor(image)
        box = torch.from_numpy(self.box_array[idx]).float()
        cls = torch.from_numpy(self.class_array[idx]).long()
        return x, box, cls


full_dataset = CocoDataset(metadata, box_array, class_array)
# Match tf.data's take(500)/skip(500): first 500 samples are validation.
val_subset = torch.utils.data.Subset(full_dataset, range(500))
train_subset = torch.utils.data.Subset(full_dataset, range(500, len(full_dataset)))

train_dataset = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=2)
val_dataset = DataLoader(val_subset, batch_size=16, num_workers=2)

#### Training the YOLO model

In [ ]:
# PyTorch: keras.ops.* -> the matching torch.* ops. divide_no_nan has no direct
# equivalent, so we clamp the denominator to avoid division by zero.

def unpack(box):
    return box[..., 0], box[..., 1], box[..., 2], box[..., 3]

def intersection(box1, box2):
    cx1, cy1, w1, h1 = unpack(box1)
    cx2, cy2, w2, h2 = unpack(box2)
    left = torch.maximum(cx1 - w1 / 2, cx2 - w2 / 2)
    bottom = torch.maximum(cy1 - h1 / 2, cy2 - h2 / 2)
    right = torch.minimum(cx1 + w1 / 2, cx2 + w2 / 2)
    top = torch.minimum(cy1 + h1 / 2, cy2 + h2 / 2)
    return torch.clamp(right - left, min=0.0) * torch.clamp(top - bottom, min=0.0)

def intersection_over_union(box1, box2):
    cx1, cy1, w1, h1 = unpack(box1)
    cx2, cy2, w2, h2 = unpack(box2)
    intersection_area = intersection(box1, box2)
    a1 = torch.clamp(w1, min=0.0) * torch.clamp(h1, min=0.0)
    a2 = torch.clamp(w2, min=0.0) * torch.clamp(h2, min=0.0)
    union_area = a1 + a2 - intersection_area
    return intersection_area / torch.clamp(union_area, min=1e-7)

In [ ]:
# PyTorch: keras.ops.* -> torch.*; keras.config.epsilon() -> a small constant.

def signed_sqrt(x):
    return torch.sign(x) * torch.sqrt(torch.abs(x) + 1e-7)

def box_loss(true, pred):
    xy_true, wh_true, conf_true = true[..., :2], true[..., 2:4], true[..., 4:]
    xy_pred, wh_pred, conf_pred = pred[..., :2], pred[..., 2:4], pred[..., 4:]
    no_object = conf_true == 0.0
    xy_error = torch.square(xy_true - xy_pred)
    wh_error = torch.square(signed_sqrt(wh_true) - signed_sqrt(wh_pred))
    iou = intersection_over_union(true, pred)
    conf_target = torch.where(no_object, torch.zeros_like(conf_pred), iou.unsqueeze(-1))
    conf_error = torch.square(conf_target - conf_pred)
    zeros2 = torch.zeros_like(xy_error)
    error = torch.cat(
        (
            torch.where(no_object, zeros2, xy_error * 5.0),
            torch.where(no_object, torch.zeros_like(wh_error), wh_error * 5.0),
            torch.where(no_object, conf_error * 0.5, conf_error),
        ),
        dim=-1,
    )
    return torch.sum(error, dim=(1, 2, 3))

In [ ]:
# PyTorch: compile() + fit() become an explicit training loop. The "box" head
# uses our box_loss; the "class" head uses CrossEntropyLoss (the torch equivalent
# of sparse_categorical_crossentropy). Note that our model already applies softmax
# to the class output, so we take its log for the NLL-style class loss instead of
# feeding logits to CrossEntropyLoss.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)


def class_loss(true, probs):
    # true: (N, grid, grid) int labels; probs: (N, grid, grid, num_labels)
    log_probs = torch.log(torch.clamp(probs, min=1e-7))
    return torch.nn.functional.nll_loss(
        log_probs.permute(0, 3, 1, 2), true, reduction="none"
    ).sum(dim=(1, 2))


def run_epoch(loader, train):
    model.train(train)
    total = 0.0
    n = 0
    for x, box_true, cls_true in loader:
        x = x.to(device)
        box_true, cls_true = box_true.to(device), cls_true.to(device)
        with torch.set_grad_enabled(train):
            preds = model(x)
            loss = (
                box_loss(box_true, preds["box"]).mean()
                + class_loss(cls_true, preds["class"]).mean()
            )
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n


for epoch in range(4):
    train_loss = run_epoch(train_dataset, train=True)
    val_loss = run_epoch(val_dataset, train=False)
    print(f"Epoch {epoch}: loss {train_loss:.4f} - val_loss {val_loss:.4f}")

In [ ]:
# PyTorch: replace model.predict() with an eval/no_grad forward pass on one item.
x, box_y, cls_y = next(iter(DataLoader(val_subset, batch_size=1)))
model.eval()
with torch.no_grad():
    preds = model(x.to(device))
boxes = preds["box"][0].cpu().numpy()
classes = np.argmax(preds["class"][0].cpu().numpy(), axis=-1)
path = metadata[0]["path"]
draw_prediction(path, boxes, classes, cutoff=0.1)

In [0]:
draw_prediction(path, boxes, classes, cutoff=None)

### Using a pretrained RetinaNet detector

In [ ]:
# PyTorch: keras.utils.get_file/load_img -> urllib download + PIL. We keep the
# image as a NumPy array with a leading batch axis to mirror the original.
url = "https://s3.us-east-1.amazonaws.com/book.keras.io/3e/seurat.jpg"
path = os.path.join(cache_dir, "seurat.jpg")
if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
image = np.array([np.array(Image.open(path).convert("RGB"))])

In [ ]:
# PyTorch: keras_hub's RetinaNet object detector (retinanet_resnet50_fpn_v2_coco)
# -> torchvision's COCO-pretrained retinanet_resnet50_fpn_v2. torchvision expects
# a list of CHW float tensors in [0, 1] and returns, per image, a dict with
# "boxes" (absolute xyxy), "labels", and "scores". We reproduce keras_hub's output
# shape (a batched dict of boxes/labels/scores/num_detections) and convert boxes to
# the book's normalized rel_xywh format so the existing drawing code still works.
from torchvision.models.detection import (
    retinanet_resnet50_fpn_v2,
    RetinaNet_ResNet50_FPN_V2_Weights,
)

weights = RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1
detector_model = retinanet_resnet50_fpn_v2(weights=weights, score_thresh=0.3)
detector_model.eval()

# torchvision's label ids index into its own COCO category list (incl. background),
# which already matches the COCO category ids used elsewhere in this notebook.

def detector_predict(image_batch, score_cutoff=0.3):
    tensors = [
        torch.from_numpy(img).permute(2, 0, 1).float() / 255.0 for img in image_batch
    ]
    with torch.no_grad():
        raw = detector_model(tensors)
    boxes_out, labels_out, scores_out, counts = [], [], [], []
    for img, out in zip(image_batch, raw):
        h, w = img.shape[:2]
        scale = max(w, h)
        keep = out["scores"] >= score_cutoff
        b = out["boxes"][keep].cpu().numpy()
        # xyxy (abs) -> rel_xywh: normalize by the longer side and pad-center like
        # the scale_box() helper used for the training data.
        rel = np.zeros_like(b)
        rel[:, 0] = b[:, 0] / scale + ((h - w) / (2 * scale) if h > w else 0)
        rel[:, 1] = b[:, 1] / scale + ((w - h) / (2 * scale) if w > h else 0)
        rel[:, 2] = (b[:, 2] - b[:, 0]) / scale
        rel[:, 3] = (b[:, 3] - b[:, 1]) / scale
        boxes_out.append(rel)
        labels_out.append(out["labels"][keep].cpu().numpy())
        scores_out.append(out["scores"][keep].cpu().numpy())
        counts.append(int(keep.sum()))
    return {
        "boxes": boxes_out,
        "labels": labels_out,
        "confidence": scores_out,
        "num_detections": counts,
    }


predictions = detector_predict(image)

In [ ]:
# PyTorch: predictions hold ragged per-image lists, so report each entry's shape.
[(k, np.shape(v[0]) if len(v) else ()) for k, v in predictions.items()]

In [0]:
predictions["boxes"][0][0]

In [ ]:
fig, ax = plt.subplots(dpi=300)
draw_image(ax, path)
num_detections = predictions["num_detections"][0]
for i in range(num_detections):
    box = predictions["boxes"][0][i]
    label = predictions["labels"][0][i]
    label_name = coco_id_to_name(label)
    draw_box(ax, box, label_name, label_to_color(label))
plt.show()